<a href="https://colab.research.google.com/github/DankHorse/Model_pretraining/blob/bert-finetuning/model_pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
import random

print("All libraries are imported!")

All libraries are imported!


In [ ]:
text = """
The cat sat on the mat.
The cat was looking at the dog.
The dog was running in the garden.
The small dog chased the cat.
The cat climbed the tree.
The bird was sitting on the tree.
The bird could see the cat and the dog.
The forest was green and beautiful.
The small creature walked through the forest.
The blue sky was above the green forest.
The sun was shining brightly.
The animals were playing together.
The dog ran through the garden.
The cat was sleeping under the tree.
The bird flew across the blue sky.
The forest was full of trees and animals.
The little cat watched the bird.
The dog sat beside the tree.
The creature walked slowly through the forest.
The animals lived happily in the green forest.
"""

print("Number of characters:", len(text))
print(text)

Number of characters: 712

The cat sat on the mat.
The cat was looking at the dog.
The dog was running in the garden.
The small dog chased the cat.
The cat climbed the tree.
The bird was sitting on the tree.
The bird could see the cat and the dog.
The forest was green and beautiful.
The small creature walked through the forest.
The blue sky was above the green forest.
The sun was shining brightly.
The animals were playing together.
The dog ran through the garden.
The cat was sleeping under the tree.
The bird flew across the blue sky.
The forest was full of trees and animals.
The little cat watched the bird.
The dog sat beside the tree.
The creature walked slowly through the forest.
The animals lived happily in the green forest.



In [ ]:
# Convert text into lowercase words
tokens = re.findall(r"\b\w+\b", text.lower())

# Create vocabulary
vocab = sorted(set(tokens))

# Add special tokens
special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[SEP]"]

vocab = special_tokens + vocab

# Create mappings
token_to_id = {token: i for i, token in enumerate(vocab)}
id_to_token = {i: token for token, i in token_to_id.items()}

print("Vocabulary size:", len(vocab))
print("\nFirst 20 tokens:")
print(vocab[:20])

Vocabulary size: 55

First 20 tokens:
['[PAD]', '[MASK]', '[CLS]', '[SEP]', 'above', 'across', 'and', 'animals', 'at', 'beautiful', 'beside', 'bird', 'blue', 'brightly', 'cat', 'chased', 'climbed', 'could', 'creature', 'dog']


In [ ]:
# Split our text into sentences
sentences = [s.strip() for s in text.strip().split("\n") if s.strip()]

# Convert each sentence into tokens and then into token IDs
encoded_sentences = []

for sentence in sentences:
    words = re.findall(r"\b\w+\b", sentence.lower())

    # Add [CLS] at the beginning and [SEP] at the end
    tokens_sentence = ["[CLS]"] + words + ["[SEP]"]

    # Convert tokens to numbers
    ids = [token_to_id[token] for token in tokens_sentence]

    encoded_sentences.append(ids)

print("Number of sentences:", len(encoded_sentences))

print("\nFirst sentence:")
print(sentences[0])

print("\nTokens:")
print(["[CLS]"] + re.findall(r"\b\w+\b", sentences[0].lower()) + ["[SEP]"])

print("\nToken IDs:")
print(encoded_sentences[0])

Number of sentences: 20

First sentence:
The cat sat on the mat.

Tokens:
['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', '[SEP]']

Token IDs:
[2, 45, 14, 36, 32, 45, 30, 3]


In [ ]:
MASK_PROBABILITY = 0.15

masked_inputs = []
masked_labels = []

for sentence_ids in encoded_sentences:

    input_ids = sentence_ids.copy()
    labels = [-100] * len(sentence_ids)

    for i in range(len(sentence_ids)):

        # Don't mask special tokens
        if sentence_ids[i] in [
            token_to_id["[CLS]"],
            token_to_id["[SEP]"],
            token_to_id["[PAD]"]
        ]:
            continue

        # Randomly choose tokens to mask
        if random.random() < MASK_PROBABILITY:

            # Save the original token as the target
            labels[i] = sentence_ids[i]

            # Replace it with [MASK]
            input_ids[i] = token_to_id["[MASK]"]

    masked_inputs.append(input_ids)
    masked_labels.append(labels)

# Display one example
print("Original:")
print(encoded_sentences[0])

print("\nMasked input:")
print(masked_inputs[0])

print("\nLabels:")
print(masked_labels[0])

Original:
[2, 45, 14, 36, 32, 45, 30, 3]

Masked input:
[2, 45, 14, 1, 32, 45, 30, 3]

Labels:
[-100, -100, -100, 36, -100, -100, -100, -100]


In [ ]:
# Find the longest sentence
max_length = max(len(x) for x in masked_inputs)

print("Maximum sequence length:", max_length)

# IDs for special tokens
pad_id = token_to_id["[PAD]"]

padded_inputs = []
padded_labels = []

for input_ids, labels in zip(masked_inputs, masked_labels):

    padding_needed = max_length - len(input_ids)

    # Pad the input with [PAD]
    padded_input = input_ids + [pad_id] * padding_needed

    # Pad the labels with -100
    padded_label = labels + [-100] * padding_needed

    padded_inputs.append(padded_input)
    padded_labels.append(padded_label)

# Convert lists into PyTorch tensors
input_tensor = torch.tensor(padded_inputs, dtype=torch.long)
label_tensor = torch.tensor(padded_labels, dtype=torch.long)

print("Input tensor shape:", input_tensor.shape)
print("Label tensor shape:", label_tensor.shape)

Maximum sequence length: 11
Input tensor shape: torch.Size([20, 11])
Label tensor shape: torch.Size([20, 11])


In [ ]:
# Model dimensions
vocab_size = len(vocab)
embedding_dim = 64

# Create the embedding layer
embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

print("Vocabulary size:", vocab_size)
print("Embedding dimension:", embedding_dim)

# Get the ID of "cat"
cat_id = token_to_id["cat"]

# Convert the ID into a tensor
cat_tensor = torch.tensor([cat_id])

# Look up its embedding
cat_embedding = embedding(cat_tensor)

print("Token ID:", cat_id)
print("Embedding shape:", cat_embedding.shape)
print("Embedding:")
print(cat_embedding)

Vocabulary size: 55
Embedding dimension: 64
Token ID: 14
Embedding shape: torch.Size([1, 64])
Embedding:
tensor([[-2.5222e-01, -2.2320e+00,  3.4923e-04, -3.8986e-01, -5.9543e-01,
         -5.2975e-01,  2.3829e+00, -7.2681e-01, -5.2042e-01,  3.4855e-01,
         -1.6720e+00,  8.6347e-01, -8.8562e-01, -4.1522e-01,  2.5925e+00,
          8.4637e-01,  3.2000e+00, -2.9940e-01, -2.4493e-01, -1.1774e+00,
          6.6713e-02, -6.0016e-01,  7.5810e-01,  5.3970e-01, -2.1172e-01,
          8.2127e-01,  8.5198e-01,  3.6622e-02, -1.3482e+00,  5.5339e-01,
         -1.6695e+00,  9.1800e-01,  6.1254e-01, -2.0996e+00,  1.4538e-01,
          5.4724e-01,  1.0773e+00,  1.4976e+00, -1.7611e-01, -1.2696e+00,
          1.0799e+00, -1.0818e+00, -6.8888e-01, -1.6191e-02,  6.7656e-01,
         -3.4534e-01,  4.7317e-01,  7.9370e-01, -7.8833e-01, -2.5626e-01,
          1.0092e+00, -5.1117e-01, -5.5177e-01, -3.6752e-01, -7.0028e-01,
         -1.3525e+00, -1.5329e+00,  5.5710e-01, -2.5781e-01, -1.5590e-01,
       

In [ ]:
# Maximum number of positions our model can handle
max_position = max_length

# Create a positional embedding layer
position_embedding = nn.Embedding(
    num_embeddings=max_position,
    embedding_dim=embedding_dim
)

# Create position IDs: 0, 1, 2, ..., max_length-1
position_ids = torch.arange(max_position)

print("Position IDs:")
print(position_ids)

print("\nPosition embedding shape:")
print(position_embedding(position_ids).shape)



# Take our input tensor
# Shape: [batch_size, sequence_length]
batch_size, sequence_length = input_tensor.shape

# Token embeddings
token_embeddings = embedding(input_tensor)

# Position IDs for this sequence
positions = torch.arange(sequence_length)

# Position embeddings
pos_embeddings = position_embedding(positions)

# Add token and position embeddings
x = token_embeddings + pos_embeddings

print("Input tensor shape:", input_tensor.shape)
print("Token embeddings shape:", token_embeddings.shape)
print("Position embeddings shape:", pos_embeddings.shape)
print("Combined representation shape:", x.shape)

Position IDs:
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

Position embedding shape:
torch.Size([11, 64])
Input tensor shape: torch.Size([20, 11])
Token embeddings shape: torch.Size([20, 11, 64])
Position embeddings shape: torch.Size([11, 64])
Combined representation shape: torch.Size([20, 11, 64])


In [ ]:
# Dimension of our embeddings
d_model = embedding_dim

# Create the three learnable projection layers
WQ = nn.Linear(d_model, d_model, bias=False)
WK = nn.Linear(d_model, d_model, bias=False)
WV = nn.Linear(d_model, d_model, bias=False)

# Create Q, K, V
Q = WQ(x)
K = WK(x)
V = WV(x)

print("X shape:", x.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

X shape: torch.Size([20, 11, 64])
Q shape: torch.Size([20, 11, 64])
K shape: torch.Size([20, 11, 64])
V shape: torch.Size([20, 11, 64])


In [ ]:
# Dimension of our embeddings
d_model = embedding_dim

# Create the three learnable projection layers
WQ = nn.Linear(d_model, d_model, bias=False)
WK = nn.Linear(d_model, d_model, bias=False)
WV = nn.Linear(d_model, d_model, bias=False)

# Create Q, K, V
Q = WQ(x)
K = WK(x)
V = WV(x)

print("X shape:", x.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)


# Calculate attention scores
attention_scores = torch.matmul(Q, K.transpose(-2, -1))

print("Attention scores shape:", attention_scores.shape)


attention_scores = attention_scores / math.sqrt(d_model)
#This calculates how strongly each token should pay attention to every other token.

print("Scaled attention scores shape:", attention_scores.shape)

X shape: torch.Size([20, 11, 64])
Q shape: torch.Size([20, 11, 64])
K shape: torch.Size([20, 11, 64])
V shape: torch.Size([20, 11, 64])
Attention scores shape: torch.Size([20, 11, 11])
Scaled attention scores shape: torch.Size([20, 11, 11])


In [ ]:
# Convert attention scores into probabilities
attention_weights = F.softmax(attention_scores, dim=-1)

print("Attention weights shape:", attention_weights.shape)

# Look at the attention weights for the first sentence
print("\nAttention weights for the first sentence:")
print(attention_weights[0])


# Check that each row sums to approximately 1
row_sums = attention_weights[0].sum(dim=-1)

print("\nRow sums:")
print(row_sums)

Attention weights shape: torch.Size([20, 11, 11])

Attention weights for the first sentence:
tensor([[0.0466, 0.0595, 0.0999, 0.0783, 0.2055, 0.0678, 0.0600, 0.0524, 0.1363,
         0.0351, 0.1586],
        [0.0270, 0.0887, 0.0710, 0.0614, 0.0271, 0.0487, 0.0363, 0.0824, 0.1758,
         0.1798, 0.2020],
        [0.0514, 0.0681, 0.1234, 0.0733, 0.1577, 0.0264, 0.0304, 0.0608, 0.0644,
         0.1303, 0.2137],
        [0.1082, 0.0664, 0.0544, 0.0608, 0.1893, 0.1090, 0.0424, 0.0901, 0.0933,
         0.0551, 0.1311],
        [0.1055, 0.1317, 0.0450, 0.0525, 0.0498, 0.0729, 0.1361, 0.0839, 0.0738,
         0.1173, 0.1315],
        [0.0515, 0.0799, 0.0785, 0.0987, 0.0553, 0.0603, 0.0603, 0.1447, 0.1357,
         0.1331, 0.1018],
        [0.0342, 0.0253, 0.0620, 0.0675, 0.1855, 0.0405, 0.0628, 0.1409, 0.1970,
         0.1012, 0.0831],
        [0.1278, 0.0559, 0.1289, 0.1186, 0.1041, 0.0525, 0.0958, 0.0711, 0.0619,
         0.0612, 0.1225],
        [0.0870, 0.0176, 0.2130, 0.1537, 0.0894, 0.

In [ ]:
attention_output = torch.matmul(attention_weights, V)

print("Attention weights shape:", attention_weights.shape)
print("V shape:", V.shape)
print("Attention output shape:", attention_output.shape)

print("\nAttention output for the first sentence:")
print(attention_output[0])

#So the token gets a mixture of information from other tokens, with more important tokens contributing more.
# 0.1 × V₁ + 0.2 × V₂ + 0.7 × V₃

Attention weights shape: torch.Size([20, 11, 11])
V shape: torch.Size([20, 11, 64])
Attention output shape: torch.Size([20, 11, 64])

Attention output for the first sentence:
tensor([[-4.9336e-01,  3.9458e-01, -3.2174e-01, -1.6152e-01, -8.6483e-02,
          2.8324e-02,  2.4435e-01,  2.2802e-01,  2.8711e-01, -4.6118e-02,
          5.6573e-02,  3.1208e-01,  2.3059e-01, -4.9382e-01, -2.5691e-01,
          2.5809e-01, -3.4530e-01,  4.0937e-02, -2.6810e-01,  9.2197e-01,
         -3.3959e-01,  1.1042e-01,  3.9472e-02, -6.5920e-03,  3.3725e-03,
          9.8936e-02,  1.2017e-02,  2.9418e-02,  3.2281e-01,  3.5525e-01,
          2.0697e-01,  2.5096e-02, -2.5090e-01, -1.3095e-01,  1.0763e-01,
          4.2386e-01, -1.2556e-01,  6.1929e-01,  1.0952e-01, -5.4889e-01,
          3.2841e-01,  6.2980e-02, -7.4642e-03,  3.6223e-01, -1.2263e-01,
          3.2897e-01,  1.4251e-01, -4.8968e-02,  4.4737e-01,  4.2949e-01,
          3.6229e-01,  5.3927e-01,  1.0984e-01,  4.5874e-01,  1.5259e-01,
         -1

In [ ]:
# Our attention output has learned to combine information from different tokens, but we want another neural-network layer to transform that combined information into a useful representation.

output_projection = nn.Linear(d_model, d_model)

projected_output = output_projection(attention_output)

print("Attention output shape:", attention_output.shape)
print("Projected output shape:", projected_output.shape)

# a learnable matrix multiplication used to mix/rearrange information

Attention output shape: torch.Size([20, 11, 64])
Projected output shape: torch.Size([20, 11, 64])


In [ ]:
layer_norm = nn.LayerNorm(d_model)

residual_output = x + projected_output
normalized_output = layer_norm(residual_output)

print("Original x shape:", x.shape)
print("Projected output shape:", projected_output.shape)
print("After residual connection:", residual_output.shape)
print("After LayerNorm:", normalized_output.shape)

Original x shape: torch.Size([20, 11, 64])
Projected output shape: torch.Size([20, 11, 64])
After residual connection: torch.Size([20, 11, 64])
After LayerNorm: torch.Size([20, 11, 64])


In [ ]:
feed_forward = nn.Sequential(
    nn.Linear(d_model, 256),
    nn.GELU(),
    nn.Linear(256, d_model)
)

ffn_output = feed_forward(normalized_output)

print("Input to FFN shape:", normalized_output.shape)
print("FFN output shape:", ffn_output.shape)

Input to FFN shape: torch.Size([20, 11, 64])
FFN output shape: torch.Size([20, 11, 64])


In [ ]:
second_residual = normalized_output + ffn_output

final_output = nn.LayerNorm(d_model)(second_residual)

print("Normalized attention output shape:", normalized_output.shape)
print("FFN output shape:", ffn_output.shape)
print("After second residual:", second_residual.shape)
print("Final Transformer block output:", final_output.shape)

Normalized attention output shape: torch.Size([20, 11, 64])
FFN output shape: torch.Size([20, 11, 64])
After second residual: torch.Size([20, 11, 64])
Final Transformer block output: torch.Size([20, 11, 64])


# This was the encoder block

### Prediction Layer

In [ ]:
mlm_head = nn.Linear(d_model, vocab_size)

logits = mlm_head(final_output)
# Logits = the model's raw scores for all possible answers

print("Final Transformer output shape:", final_output.shape)
print("MLM logits shape:", logits.shape)


# If your vocabulary has 10,000 words, the MLM head produces 10,000 scores for each position:

Final Transformer output shape: torch.Size([20, 11, 64])
MLM logits shape: torch.Size([20, 11, 55])


In [ ]:
loss_function = nn.CrossEntropyLoss(ignore_index=-100)
# -100 = Don't calculate loss for this position

loss = loss_function(
    logits.view(-1, vocab_size),
    label_tensor.view(-1)
)

print("Loss:", loss.item())

Loss: 4.382269382476807


# Backpropagation

In [ ]:
# We first ask PyTorch to calculate gradients:

loss.backward()

print("Gradients calculated successfully!")

Gradients calculated successfully!


In [ ]:
# Backpropagation does NOT change the weights.
# It only calculates the gradients.

# Gradients tell us how the weights should change; the optimizer actually changes them.

# The actual weight update happens with an optimizer, which we'll do next.

In [ ]:
optimizer = torch.optim.Adam(
    list(embedding.parameters()) +
    list(position_embedding.parameters()) +
    list(WQ.parameters()) +
    list(WK.parameters()) +
    list(WV.parameters()) +
    list(output_projection.parameters()) +
    list(mlm_head.parameters()) +
    list(feed_forward.parameters()) +
    list(layer_norm.parameters()),
    lr=0.001

    # A larger learning rate means bigger updates; a smaller one means smaller updates.
)

optimizer.step()

print("Optimizer step completed!")

# Use gradients to move the weights toward lower loss.

# Adam is the optimizer that uses the gradients calculated from the loss to adjust the Transformer’s trainable weights, and optimizer.step() performs that actual weight update.

Optimizer step completed!


In [ ]:
print("Vocabulary size:", vocab_size)
print("Embedding dimension:", embedding_dim)
print("Maximum sequence length:", max_length)
print("Input tensor shape:", input_tensor.shape)
print("Label tensor shape:", label_tensor.shape)

Vocabulary size: 55
Embedding dimension: 64
Maximum sequence length: 11
Input tensor shape: torch.Size([20, 11])
Label tensor shape: torch.Size([20, 11])


In [ ]:
class TinyBERT(nn.Module):
    def __init__(self, vocab_size, max_length, pad_id, d_model=64, hidden_dim=256):
        super().__init__()

        self.d_model = d_model
        self.pad_id = pad_id

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)

        self.WQ = nn.Linear(d_model, d_model)
        self.WK = nn.Linear(d_model, d_model)
        self.WV = nn.Linear(d_model, d_model)

        self.output_projection = nn.Linear(d_model, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model)
        )

        self.mlm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):

        batch_size, sequence_length = input_ids.shape

        # Token embeddings
        token_embeddings = self.token_embedding(input_ids)

        # Position embeddings
        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        )

        position_embeddings = self.position_embedding(positions)

        # Combine token + position embeddings
        x = token_embeddings + position_embeddings

        # Q, K, V
        Q = self.WQ(x)
        K = self.WK(x)
        V = self.WV(x)

        # Attention scores
        attention_scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.d_model)

        # Padding mask
        padding_mask = (
            input_ids == self.pad_id
        ).unsqueeze(1)

        # Ignore PAD tokens
        attention_scores = attention_scores.masked_fill(
            padding_mask,
            -1e9
        )

        # Attention weights
        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        # Attention output
        attention_output = torch.matmul(
            attention_weights,
            V
        )

        # Projection
        attention_output = self.output_projection(
            attention_output
        )

        # Residual + LayerNorm
        x = self.norm1(x + attention_output)

        # FFN
        ffn_output = self.feed_forward(x)

        # Second residual + LayerNorm
        x = self.norm2(x + ffn_output)

        # MLM prediction
        logits = self.mlm_head(x)

        return logits


# Create a fresh model
model = TinyBERT(
        # does NOT train the model.
# It only creates the neural network and initializes its parameters.

    vocab_size=vocab_size,
        # vocab size = 55

    max_length=max_length,
        # 11 tokens

    pad_id=pad_id,
    d_model=64,
    hidden_dim=256
        # means the Feed-Forward Network temporarily expands each token's representation from:
    # Because the larger intermediate space allows the network to perform more complex transformations.
)

print(model)
# shows the architecture of the neural network

TinyBERT(
  (token_embedding): Embedding(55, 64)
  (position_embedding): Embedding(11, 64)
  (WQ): Linear(in_features=64, out_features=64, bias=True)
  (WK): Linear(in_features=64, out_features=64, bias=True)
  (WV): Linear(in_features=64, out_features=64, bias=True)
  (output_projection): Linear(in_features=64, out_features=64, bias=True)
  (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (feed_forward): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=64, bias=True)
  )
  (mlm_head): Linear(in_features=64, out_features=55, bias=True)
)


In [ ]:
pad_id = token_to_id["[PAD]"]

padding_mask = (input_tensor == pad_id)

print("PAD token ID:", pad_id)
print("\nPadding mask for first sentence:")
print(padding_mask[0])

PAD token ID: 0

Padding mask for first sentence:
tensor([False, False, False, False, False, False, False, False,  True,  True,
         True])


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
input_tensor = input_tensor.to(device)
label_tensor = label_tensor.to(device)

print("Using device:", device)

# Test the model
logits = model(input_tensor)

print("Input shape:", input_tensor.shape)
print("Output shape:", logits.shape)

# 20 → sentences
# 11 → token positions
# 55 → vocabulary size

# So for every token position, our model produces scores for all 55 possible vocabulary tokens.

Using device: cuda
Input shape: torch.Size([20, 11])
Output shape: torch.Size([20, 11, 55])


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

print("Optimizer created successfully!")
print("Number of trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

Optimizer created successfully!
Number of trainable parameters: 57783


In [ ]:
print("Vocabulary size:", vocab_size)
print("Maximum length:", max_length)

for name, param in model.named_parameters():
    print(name, param.numel())

Vocabulary size: 55
Maximum length: 11
token_embedding.weight 3520
position_embedding.weight 704
WQ.weight 4096
WQ.bias 64
WK.weight 4096
WK.bias 64
WV.weight 4096
WV.bias 64
output_projection.weight 4096
output_projection.bias 64
norm1.weight 64
norm1.bias 64
norm2.weight 64
norm2.bias 64
feed_forward.0.weight 16384
feed_forward.0.bias 256
feed_forward.2.weight 16384
feed_forward.2.bias 64
mlm_head.weight 3520
mlm_head.bias 55


In [ ]:
loss_function = nn.CrossEntropyLoss(
    ignore_index=-100
)

# Make predictions
logits = model(input_tensor)

# Calculate loss
loss = loss_function(
    logits.view(-1, vocab_size),
    label_tensor.view(-1)
)

print("Initial loss:", loss.item())

Initial loss: 4.199169158935547


# Training

In [ ]:
# Put the model in training mode
model.train()

# Clear old gradients
optimizer.zero_grad()

# Forward pass
logits = model(input_tensor)

# Calculate loss
loss = loss_function(
    logits.view(-1, vocab_size),
    label_tensor.view(-1)
)

# Backpropagation
loss.backward()

# Update model weights
optimizer.step()

print("Loss after one training step:", loss.item())

Loss after one training step: 4.199169158935547


In [ ]:
epochs = 100

for epoch in range(epochs):

    model.train()

    # Clear previous gradients
    optimizer.zero_grad()

    # Forward pass
    logits = model(input_tensor)

    # Calculate loss
    loss = loss_function(
        logits.view(-1, vocab_size),
        label_tensor.view(-1)
    )

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Print progress
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 10/100, Loss: 2.2477
Epoch 20/100, Loss: 1.2258
Epoch 30/100, Loss: 0.7724
Epoch 40/100, Loss: 0.4934
Epoch 50/100, Loss: 0.2662
Epoch 60/100, Loss: 0.1362
Epoch 70/100, Loss: 0.0824
Epoch 80/100, Loss: 0.0565
Epoch 90/100, Loss: 0.0429
Epoch 100/100, Loss: 0.0347


In [ ]:
model.eval()

# Choose a sentence
test_sentence = "The cat sat on the [MASK]."

# Extract words and preserve [MASK]
test_words = re.findall(r"\[MASK\]|\b\w+\b", test_sentence)

# Convert normal words to lowercase, but keep [MASK] unchanged
test_words = [
    token if token == "[MASK]" else token.lower()
    for token in test_words
]

tokens_test = ["[CLS]"] + test_words + ["[SEP]"]

# Convert tokens to IDs
test_ids = [
    token_to_id[token]
    for token in tokens_test
]

# Pad to maximum length
padding_needed = max_length - len(test_ids)
test_ids = test_ids + [pad_id] * padding_needed

# Convert to tensor
test_tensor = torch.tensor(
    [test_ids],
    dtype=torch.long
).to(device)

# Get predictions
with torch.no_grad():
    logits = model(test_tensor)

# Find the MASK position
mask_id = token_to_id["[MASK]"]
mask_position = test_ids.index(mask_id)

# Get predictions at MASK position
mask_logits = logits[0, mask_position]

# Convert logits to probabilities
probabilities = F.softmax(mask_logits, dim=-1)

# Get top 5 predictions
top_probs, top_ids = torch.topk(
    probabilities,
    k=5
)

print("Input:", test_sentence)
print("\nTop predictions:")

for prob, token_id in zip(top_probs, top_ids):
    print(
        f"{id_to_token[token_id.item()]} "
        f"→ {prob.item():.4f}"
    )

Input: The cat sat on the [MASK].

Top predictions:
the → 0.9759
shining → 0.0030
bird → 0.0027
was → 0.0024
tree → 0.0014


In [ ]:
def create_masked_batch(encoded_sentences, mask_probability=0.15):
  # it randomly chooses different words to hide.

    masked_inputs = []
    masked_labels = []

    for sentence_ids in encoded_sentences:

        input_ids = sentence_ids.copy()
        labels = [-100] * len(sentence_ids)

        for i in range(len(sentence_ids)):

            # Don't mask special tokens
            if sentence_ids[i] in [
                token_to_id["[CLS]"],
                token_to_id["[SEP]"],
                token_to_id["[PAD]"]
            ]:
                continue

            # Randomly select tokens to mask
            if random.random() < mask_probability:

                labels[i] = sentence_ids[i]
                input_ids[i] = token_to_id["[MASK]"]

        masked_inputs.append(input_ids)
        masked_labels.append(labels)

    # Convert to tensors
    input_tensor = torch.tensor(
        masked_inputs,
        dtype=torch.long
    )

    label_tensor = torch.tensor(
        masked_labels,
        dtype=torch.long
    )

    return input_tensor, label_tensor


print("Dynamic masking function created!")

Dynamic masking function created!


In [ ]:
model = TinyBERT(
    vocab_size=vocab_size,
    max_length=max_length,
    pad_id=pad_id,
    d_model=64,
    hidden_dim=256
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

loss_function = nn.CrossEntropyLoss(
    ignore_index=-100
)

print("Fresh model created!")
print("Using device:", device)

Fresh model created!
Using device: cuda


In [ ]:
def create_masked_batch(encoded_sentences, mask_probability=0.15):
  # pytorch need sentences to be of same length and hence padding is necessary

    masked_inputs = []
    masked_labels = []

    for sentence_ids in encoded_sentences:

        input_ids = sentence_ids.copy()
        labels = [-100] * len(sentence_ids)

        for i in range(len(sentence_ids)):

            # Don't mask special tokens
            if sentence_ids[i] in [
                token_to_id["[CLS]"],
                token_to_id["[SEP]"],
                token_to_id["[PAD]"]
            ]:
                continue

            # Randomly select tokens to mask
            if random.random() < mask_probability:

                labels[i] = sentence_ids[i]
                input_ids[i] = token_to_id["[MASK]"]

        # Add padding
        padding_needed = max_length - len(input_ids)

        input_ids += [pad_id] * padding_needed
        labels += [-100] * padding_needed

        masked_inputs.append(input_ids)
        masked_labels.append(labels)

    # Convert to tensors
    input_tensor = torch.tensor(
        masked_inputs,
        dtype=torch.long
    )

    label_tensor = torch.tensor(
        masked_labels,
        dtype=torch.long
    )

    return input_tensor, label_tensor


print("Dynamic masking function created successfully!")

Dynamic masking function created successfully!


In [ ]:
epochs = 100

for epoch in range(epochs):

    model.train()

    # Create a new masking pattern every epoch
    train_input, train_labels = create_masked_batch(
        encoded_sentences,
        mask_probability=0.15
        # the model gets a fresh random masking pattern.
    )

    train_input = train_input.to(device)
    train_labels = train_labels.to(device)

    # Forward pass
    logits = model(train_input)

    # Calculate loss
    loss = loss_function(
        logits.view(-1, vocab_size),
        train_labels.view(-1)
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 10/100, Loss: 3.7524
Epoch 20/100, Loss: 2.8879
Epoch 30/100, Loss: 2.3633
Epoch 40/100, Loss: 2.1287
Epoch 50/100, Loss: 2.4224
Epoch 60/100, Loss: 2.0559
Epoch 70/100, Loss: 2.2775
Epoch 80/100, Loss: 1.3471
Epoch 90/100, Loss: 2.1855
Epoch 100/100, Loss: 1.5583


In [ ]:
train_input, train_labels = create_masked_batch(
    encoded_sentences,
    mask_probability=0.15
)

num_masked = (train_labels != -100).sum().item()

print("Input shape:", train_input.shape)
print("Number of masked tokens:", num_masked)
print("Total tokens:", train_labels.numel())
print(
    "Masking percentage:",
    f"{100 * num_masked / train_labels.numel():.2f}%"
)

Input shape: torch.Size([20, 11])
Number of masked tokens: 25
Total tokens: 220
Masking percentage: 11.36%


In [ ]:
def create_masked_batch(encoded_sentences, mask_probability=0.15):

    masked_inputs = []
    masked_labels = []

    for sentence_ids in encoded_sentences:

        input_ids = sentence_ids.copy()
        labels = [-100] * len(sentence_ids)

        # Find tokens that are allowed to be masked
        candidate_positions = [
            i for i, token_id in enumerate(sentence_ids)
            if token_id not in [
                token_to_id["[CLS]"],
                token_to_id["[SEP]"],
                token_to_id["[PAD]"]
            ]
        ]

        # Number of tokens to mask
        num_to_mask = max(
            1,
            int(len(candidate_positions) * mask_probability)
        )

        # Randomly select positions
        mask_positions = random.sample(
            candidate_positions,
            num_to_mask
        )

        # Apply masking
        for position in mask_positions:

            labels[position] = sentence_ids[position]
            input_ids[position] = token_to_id["[MASK]"]

        # Pad to max_length
        padding_needed = max_length - len(input_ids)

        input_ids += [pad_id] * padding_needed
        labels += [-100] * padding_needed

        masked_inputs.append(input_ids)
        masked_labels.append(labels)

    # Convert to tensors
    input_tensor = torch.tensor(
        masked_inputs,
        dtype=torch.long
    )

    label_tensor = torch.tensor(
        masked_labels,
        dtype=torch.long
    )

    return input_tensor, label_tensor


print("Improved masking function created!")

Improved masking function created!


In [ ]:
train_input, train_labels = create_masked_batch(
    encoded_sentences,
    mask_probability=0.15
)

num_masked = (train_labels != -100).sum().item()

print("Input shape:", train_input.shape)
print("Label shape:", train_labels.shape)
print("Number of masked tokens:", num_masked)

print("\nFirst sentence - masked input:")
print(train_input[0])

print("\nFirst sentence - labels:")
print(train_labels[0])

Input shape: torch.Size([20, 11])
Label shape: torch.Size([20, 11])
Number of masked tokens: 20

First sentence - masked input:
tensor([ 2, 45, 14, 36, 32,  1, 30,  3,  0,  0,  0])

First sentence - labels:
tensor([-100, -100, -100, -100, -100,   45, -100, -100, -100, -100, -100])


In [ ]:
epochs = 200

for epoch in range(epochs):

    model.train()

    # Create a fresh masking pattern
    train_input, train_labels = create_masked_batch(
        encoded_sentences,
        mask_probability=0.15
    )

    train_input = train_input.to(device)
    train_labels = train_labels.to(device)

    # Forward pass
    logits = model(train_input)

    # Calculate loss
    loss = loss_function(
        logits.view(-1, vocab_size),
        train_labels.view(-1)
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 20/200, Loss: 1.4315
Epoch 40/200, Loss: 1.3150
Epoch 60/200, Loss: 0.8240
Epoch 80/200, Loss: 0.7422
Epoch 100/200, Loss: 0.4968
Epoch 120/200, Loss: 0.4165
Epoch 140/200, Loss: 0.4647
Epoch 160/200, Loss: 0.2620
Epoch 180/200, Loss: 0.1223
Epoch 200/200, Loss: 0.5058


In [ ]:
model.eval()

test_sentence = "The cat sat on the [MASK]."

# Preserve [MASK] while lowercasing normal words
test_words = re.findall(r"\[MASK\]|\b\w+\b", test_sentence)

test_words = [
    token if token == "[MASK]" else token.lower()
    for token in test_words
]

tokens_test = ["[CLS]"] + test_words + ["[SEP]"]

# Convert tokens to IDs
test_ids = [
    token_to_id[token]
    for token in tokens_test
]

# Pad to max length
padding_needed = max_length - len(test_ids)

test_ids += [pad_id] * padding_needed

# Convert to tensor
test_tensor = torch.tensor(
    [test_ids],
    dtype=torch.long
).to(device)

# Make prediction
with torch.no_grad():
    logits = model(test_tensor)

# Find MASK position
mask_id = token_to_id["[MASK]"]
mask_position = test_ids.index(mask_id)

# Get logits for MASK
mask_logits = logits[0, mask_position]

# Convert logits to probabilities
probabilities = F.softmax(mask_logits, dim=-1)

# Get top 5 predictions
top_probs, top_ids = torch.topk(
    probabilities,
    k=5
)

print("Input:", test_sentence)

print("\nTop predictions:")

for prob, token_id in zip(top_probs, top_ids):
    print(
        f"{id_to_token[token_id.item()]} "
        f"→ {prob.item():.4f}"
    )

Input: The cat sat on the [MASK].

Top predictions:
mat → 0.4860
tree → 0.1930
the → 0.1154
bird → 0.0446
garden → 0.0244


# Testing

In [ ]:
test_sentences = [
    "The cat sat on the [MASK].",
    "The bird was sitting on the [MASK].",
    "The dog was running in the [MASK].",
    "The forest was [MASK] and beautiful."
]

model.eval()

for test_sentence in test_sentences:

    # Tokenize while preserving [MASK]
    test_words = re.findall(
        r"\[MASK\]|\b\w+\b",
        test_sentence
    )

    test_words = [
        token if token == "[MASK]" else token.lower()
        for token in test_words
    ]

    tokens_test = ["[CLS]"] + test_words + ["[SEP]"]

    # Convert to IDs
    test_ids = [
        token_to_id[token]
        for token in tokens_test
    ]

    # Pad
    padding_needed = max_length - len(test_ids)
    test_ids += [pad_id] * padding_needed

    test_tensor = torch.tensor(
        [test_ids],
        dtype=torch.long
    ).to(device)

    # Prediction
    with torch.no_grad():
        logits = model(test_tensor)

    # Find MASK
    mask_position = test_ids.index(mask_id)

    mask_logits = logits[0, mask_position]

    probabilities = F.softmax(
        mask_logits,
        dim=-1
    )

    top_probs, top_ids = torch.topk(
        probabilities,
        k=3
    )

    print("\nInput:", test_sentence)
    print("Predictions:")

    for prob, token_id in zip(top_probs, top_ids):
        print(
            f"  {id_to_token[token_id.item()]} "
            f"→ {prob.item():.4f}"
        )


Input: The cat sat on the [MASK].
Predictions:
  mat → 0.4860
  tree → 0.1930
  the → 0.1154

Input: The bird was sitting on the [MASK].
Predictions:
  tree → 0.8310
  and → 0.0406
  forest → 0.0333

Input: The dog was running in the [MASK].
Predictions:
  garden → 0.6948
  tree → 0.1094
  and → 0.0719

Input: The forest was [MASK] and beautiful.
Predictions:
  green → 0.5166
  full → 0.2412
  shining → 0.0325


# Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model=64,
        num_heads=4,
        pad_id=0
    ):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.pad_id = pad_id

        # Q, K, V projections
        self.WQ = nn.Linear(d_model, d_model)
        self.WK = nn.Linear(d_model, d_model)
        self.WV = nn.Linear(d_model, d_model)

        # Final projection
        self.output_projection = nn.Linear(
            d_model,
            d_model
        )

    def forward(self, x, input_ids):

        batch_size, seq_length, _ = x.shape

        # Create Q, K, V
        Q = self.WQ(x)
        K = self.WK(x)
        V = self.WV(x)

        # Split into heads
        Q = Q.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Attention scores
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        # Create padding mask
        padding_mask = (
            input_ids == self.pad_id
        ).unsqueeze(1).unsqueeze(2)

        # Mask PAD tokens
        scores = scores.masked_fill(
            padding_mask,
            -1e9
        )

        # Attention weights
        weights = F.softmax(
            scores,
            dim=-1
        )

        # Weighted sum of V
        attention = torch.matmul(
            weights,
            V
        )

        # Combine heads
        attention = attention.transpose(1, 2)

        attention = attention.contiguous().view(
            batch_size,
            seq_length,
            self.d_model
        )

        # Final projection
        output = self.output_projection(
            attention
        )

        return output


multi_head_attention = MultiHeadAttention(
    d_model=64,
    num_heads=4,
    pad_id=pad_id
)

test_output = multi_head_attention(
    x,
    input_tensor
)

print("Input shape:", x.shape)
print("Output shape:", test_output.shape)

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

In [ ]:
print("Device variable:", device)
print("x device:", x.device)
print("input_tensor device:", input_tensor.device)

Device variable: cuda
x device: cpu
input_tensor device: cuda:0


In [ ]:
x = x.to(device)
multi_head_attention = multi_head_attention.to(device)

test_output = multi_head_attention(
    x,
    input_tensor
)

print("x device:", x.device)
print("input_tensor device:", input_tensor.device)
print("Output shape:", test_output.shape)

x device: cuda:0
input_tensor device: cuda:0
Output shape: torch.Size([20, 11, 64])


In [ ]:
class TinyBERTMultiHead(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_length,
        pad_id,
        d_model=64,
        num_heads=4,
        hidden_dim=256
    ):
        super().__init__()

        self.d_model = d_model
        self.pad_id = pad_id

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Position embeddings
        self.position_embedding = nn.Embedding(
            max_length,
            d_model
        )

        # Multi-head attention
        self.attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
            pad_id=pad_id
        )

        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Feed-forward network
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model)
        )

        # MLM prediction head
        self.mlm_head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, input_ids):

        batch_size, sequence_length = input_ids.shape

        # Token embeddings
        token_embeddings = self.token_embedding(input_ids)

        # Position embeddings
        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        )

        position_embeddings = self.position_embedding(
            positions
        )

        # Combine token + position information
        x = token_embeddings + position_embeddings

        # Multi-head self-attention
        attention_output = self.attention(
            x,
            input_ids
        )

        # First residual connection + LayerNorm
        x = self.norm1(
            x + attention_output
        )

        # Feed-forward network
        ffn_output = self.feed_forward(x)

        # Second residual connection + LayerNorm
        x = self.norm2(
            x + ffn_output
        )

        # MLM prediction
        logits = self.mlm_head(x)

        return logits


model = TinyBERTMultiHead(
    vocab_size=vocab_size,
    max_length=max_length,
    pad_id=pad_id,
    d_model=64,
    num_heads=4,
    hidden_dim=256
).to(device)

print(model)

TinyBERTMultiHead(
  (token_embedding): Embedding(55, 64)
  (position_embedding): Embedding(11, 64)
  (attention): MultiHeadAttention(
    (WQ): Linear(in_features=64, out_features=64, bias=True)
    (WK): Linear(in_features=64, out_features=64, bias=True)
    (WV): Linear(in_features=64, out_features=64, bias=True)
    (output_projection): Linear(in_features=64, out_features=64, bias=True)
  )
  (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (feed_forward): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=64, bias=True)
  )
  (mlm_head): Linear(in_features=64, out_features=55, bias=True)
)


In [ ]:
logits = model(input_tensor)

print("Input shape:", input_tensor.shape)
print("Logits shape:", logits.shape)
print("Device:", logits.device)

Input shape: torch.Size([20, 11])
Logits shape: torch.Size([20, 11, 55])
Device: cuda:0


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

loss_function = nn.CrossEntropyLoss(
    ignore_index=-100
)

print("Optimizer created!")
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters()
        if p.requires_grad)
)

Optimizer created!
Trainable parameters: 57783


In [ ]:
model.eval()

train_input, train_labels = create_masked_batch(
    encoded_sentences,
    mask_probability=0.15
)

train_input = train_input.to(device)
train_labels = train_labels.to(device)

with torch.no_grad():
    logits = model(train_input)

loss = loss_function(
    logits.view(-1, vocab_size),
    train_labels.view(-1)
)

print("Initial loss:", loss.item())

Initial loss: 4.359261512756348


In [ ]:
epochs = 200

for epoch in range(epochs):

    model.train()

    # Create fresh masked data
    train_input, train_labels = create_masked_batch(
        encoded_sentences,
        mask_probability=0.15
    )

    # Move data to GPU
    train_input = train_input.to(device)
    train_labels = train_labels.to(device)

    # Forward pass
    logits = model(train_input)

    # Calculate loss
    loss = loss_function(
        logits.view(-1, vocab_size),
        train_labels.view(-1)
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Update model weights
    optimizer.step()

    # Print progress
    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 20/200, Loss: 2.7807
Epoch 40/200, Loss: 2.3254
Epoch 60/200, Loss: 2.1133
Epoch 80/200, Loss: 1.8059
Epoch 100/200, Loss: 1.6339
Epoch 120/200, Loss: 0.6715
Epoch 140/200, Loss: 0.8239
Epoch 160/200, Loss: 0.5417
Epoch 180/200, Loss: 0.6388
Epoch 200/200, Loss: 0.3637


In [ ]:
model.eval()

test_sentence = "The cat sat on the [MASK]."

test_words = re.findall(
    r"\[MASK\]|\b\w+\b",
    test_sentence
)

test_words = [
    token if token == "[MASK]" else token.lower()
    for token in test_words
]

tokens_test = ["[CLS]"] + test_words + ["[SEP]"]

test_ids = [
    token_to_id[token]
    for token in tokens_test
]

padding_needed = max_length - len(test_ids)

test_ids += [pad_id] * padding_needed

test_tensor = torch.tensor(
    [test_ids],
    dtype=torch.long
).to(device)

with torch.no_grad():
    logits = model(test_tensor)

mask_id = token_to_id["[MASK]"]
mask_position = test_ids.index(mask_id)

mask_logits = logits[0, mask_position]

probabilities = F.softmax(
    mask_logits,
    dim=-1
)

top_probs, top_ids = torch.topk(
    probabilities,
    k=5
)

print("Input:", test_sentence)
print("\nTop predictions:")

for prob, token_id in zip(top_probs, top_ids):
    print(
        f"{id_to_token[token_id.item()]} "
        f"→ {prob.item():.4f}"
    )

Input: The cat sat on the [MASK].

Top predictions:
mat → 0.5993
tree → 0.0814
dog → 0.0370
sat → 0.0212
beautiful → 0.0196
